In [1]:
# Установка (один раз)
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    AutoModel
)
from datasets import Dataset
import joblib
from sklearn.metrics import accuracy_score, f1_score

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
from enum import Enum


class Priority(Enum):
    LOW = 0
    MEDIUM = 1
    HIGH = 2
    CRITICAL = 3


class Category(Enum):
    PAYMENT = 0
    DELIVERY = 1
    TECH = 2
    PRODUCT = 3
    SPAM = 4

In [4]:
# Проверяем Видеокарту
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Видеопамять: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.1f} ГБ")

Используем устройство: cuda
GPU: Tesla V100-SXM2-32GB
Видеопамять: 31.7 ГБ


In [5]:
df = pd.read_csv('FullDataset3.csv')

In [6]:
df

,text,category_value,priority_value
0,"Добрый вечер, я столкнулся с проблемой при опл...",0,0
1,"Приветствую, у меня возникла странная ситуация...",0,0
2,"Приветствую, при попытке оплаты возникает ошиб...",0,0
3,"Приветствую, система списала деньги, но заказ ...",0,0
4,"Здравствуйте, я столкнулся с проблемой при опл...",0,0
...,...,...,...
5854,"Здравствуйте! Подскажите, совместим ли данный ...",3,1
5855,Здравствуйте. Я хочу оформить возврат сложной ...,3,2
5856,ВНИМАНИЕ! Ваша банковская карта была выбрана д...,4,0
5857,Здравствуйте! Мы представляем сервис автоматиз...,4,0


# Настройки для обучения

In [7]:
# Настройки
train_size = 0.8
val_to_test_size = 0.5 # Делим test и validation пополам

# Разделение датасета на train, validation и test и редактирование датасета

In [8]:
train_df, temp_df = train_test_split(
    df, 
    test_size=1-train_size,
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=val_to_test_size,
    random_state=42,
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 4687, Val: 586, Test: 586


In [9]:
train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Токенизирование датасета

In [10]:
# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )
    # Добавляем метки обратно, чтобы они были доступны в датасете
    tokenized_inputs['category_value'] = examples['category_value']
    tokenized_inputs['priority_value'] = examples['priority_value']
    return tokenized_inputs

In [11]:
# Токенизируем тексты
train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4687 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

In [12]:
train_dataset

Dataset({
    features: ['text', 'category_value', 'priority_value', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4687
})

In [13]:
print(min(train_dataset["priority_value"]), max(train_dataset["priority_value"]))
print(min(train_dataset["category_value"]), max(train_dataset["category_value"]))

0 3
0 4


# Код модели

In [31]:
category_coefficient = .6
priority_coefficient = 2

In [32]:
penalty_matrix = torch.tensor([
    [0, 1, 4, 4],
    [1, 0, 1, 3],
    [2, 1, 0, 1],
    [4, 4, 2, 0],
], dtype=torch.float, device=device)

def custom_priority_loss(logits, labels):
    probs = torch.softmax(logits, dim=-1)
    penalties = penalty_matrix[labels]
    return (probs * penalties).sum(dim=1).mean()

In [33]:
import torch
from transformers import AutoModel, Trainer

# Класс двухголовой модели
class RuBERTMultiTask(torch.nn.Module):
    def __init__(self, model_name, num_categories=5, num_priorities=4, *args, **kwargs):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Две независимые головы
        self.category_head = torch.nn.Linear(hidden_size, num_categories)
        self.priority_head = torch.nn.Linear(hidden_size, num_priorities)

    def forward(self, input_ids, attention_mask, token_type_ids=None, *args, **kwargs): # Добавили token_type_ids
        # Общий кодировщик
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Берем CLS-токен (вектор всего предложения)
        pooled = outputs.last_hidden_state[:, 0, :]

        # Два выхода
        category_logits = self.category_head(pooled)
        priority_logits = self.priority_head(pooled)

        return category_logits, priority_logits

In [34]:
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Достаём правильные ответы
        category_labels = inputs.pop('category_value').to(device)
        priority_labels = inputs.pop('priority_value').to(device)

        # Forward pass — модель выдаёт оба предсказания
        category_logits, priority_logits = model(**inputs)

        # Считаем ошибку для каждой головы
        loss_cat = torch.nn.CrossEntropyLoss()(category_logits, category_labels)  # Ошибка головы А
        loss_prior = custom_priority_loss(priority_logits, priority_labels)       # Ошибка головы Б

        # Используем глобальные коэффициенты, определенные ранее
        total_loss = category_coefficient * loss_cat + priority_coefficient * loss_prior

        return (total_loss, (category_logits, priority_logits)) if return_outputs else total_loss

    # Упорядочиваем тензоры в памяти, перед сохранение модели
    # Если не делать - будет ошибка во время обучения
    def save_model(self, output_dir=None, _internal_call=False):
        for param in self.model.parameters():
            if not param.data.is_contiguous():
                param.data = param.data.contiguous()
        super().save_model(output_dir, _internal_call)

In [35]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class MultiTaskDataCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        category_labels = [f['category_value'] for f in features]
        priority_labels = [f['priority_value'] for f in features]

        # Создаем список словарей, содержащих только токенизированные входы (input_ids, attention_mask, token_type_ids)
        # Это необходимо, чтобы tokenizer.pad корректно обработал их, игнорируя другие ключи из датасета.
        batch_for_padding = []
        for item in features:
            temp_dict = {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
            }
            # Включаем token_type_ids, если они есть (например, для токенизаторов BERT)
            if "token_type_ids" in item:
                temp_dict["token_type_ids"] = item["token_type_ids"]
            batch_for_padding.append(temp_dict)

        batch = self.tokenizer.pad(
            batch_for_padding,
            return_tensors="pt",
        )

        # Добавляем метки обратно в батч
        batch['category_value'] = torch.tensor(category_labels, dtype=torch.long)
        batch['priority_value'] = torch.tensor(priority_labels, dtype=torch.long)

        return batch

In [36]:
# 4. Загружаем токенизатор и модель
model = RuBERTMultiTask("DeepPavlov/rubert-base-cased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    remove_unused_columns=False,
    num_train_epochs=10,
    logging_dir='./logs',
    fp16=True,  # если GPU поддерживает
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [38]:
 data_collator = MultiTaskDataCollator(tokenizer=tokenizer)

# # Инициализация Trainer с вашим custom DataCollator
trainer = MultiTaskTrainer(
    model=model,  # Ваша модель RuBERTMultiTask
    args=training_args,
    train_dataset=train_dataset, # Ваш тренировочный Dataset
    eval_dataset=validation_dataset, # Ваш валидационный Dataset
    data_collator=data_collator,
)

In [39]:
train_dataset

Dataset({
    features: ['text', 'category_value', 'priority_value', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4687
})

In [40]:
trainer.train()

Step,Training Loss
500,2.318964
1000,1.784548
1500,1.541021
2000,1.500751
2500,1.394388


TrainOutput(global_step=2930, training_loss=1.6509144649570713, metrics={'train_runtime': 352.695, 'train_samples_per_second': 132.891, 'train_steps_per_second': 8.307, 'total_flos': 0.0, 'train_loss': 1.6509144649570713, 'epoch': 10.0})

# Тестирование обученой модели

In [41]:
def predict(string: str):
    input_tokenized = tokenizer(
        string,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    input_tokenized = {k: v.to(device) for k, v in input_tokenized.items()}
    # Предсказание
    with torch.no_grad():
        category_logits, priority_logits = model(**input_tokenized)
    
    # Перевод в вероятности
    category_probs = torch.softmax(category_logits, dim=-1)
    priority_probs = torch.softmax(priority_logits, dim=-1)
    
    # Индексы классов
    category_pred = torch.argmax(category_probs, dim=-1).item()
    priority_pred = torch.argmax(priority_probs, dim=-1).item()
    return f"Category: {Category(category_pred).name}, Priority: {Priority(priority_pred).name}"

In [42]:
predict("Здравствуйте, ")

'Category: SPAM, Priority: LOW'

In [43]:
for i, example in zip(range(10), test_dataset):
    print(f"{i}.) '{example['text']}': {predict(example['text'])}")

0.) 'эксклюзивное предложение только сегодня эксклюзивное предложение только сегодня предлагаем быстрый заработок без вложений эксклюзивное предложение только сегодня подтвердите данные аккаунта срочно получите бонус прямо сейчас вы выиграли приз перейдите по ссылке вы выиграли приз перейдите по ссылке получите бонус прямо сейчас подтвердите данные аккаунта срочно подтвердите данные аккаунта срочно эксклюзивное предложение только сегодня': Category: SPAM, Priority: MEDIUM
1.) 'Я хочу оформить возврат технически сложного товара (кофемашина), так как она не подошла мне по габаритам. Упаковку не вскрывал, все пломбы на месте. Какая процедура возврата в этом случае?': Category: PRODUCT, Priority: MEDIUM
2.) 'Заработок на лайках от 5к в день! Работа для студентов и мам в декрете. Никаких вложений, чекай линк.': Category: SPAM, Priority: LOW
3.) 'Почему у меня в ЛК цена на стафф меняется каждые пять минут? Это байт такой или че? Я только нажал 'оплатить', а ценник прыгнул на косарь вверх. Не

In [44]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [45]:
def evaluate_model(model, dataloader, device="cpu"):
    model.eval()
    model.to(device)

    total_loss = 0
    total_samples = 0

    correct_cat = 0
    correct_pr = 0

    all_cat_preds = []
    all_cat_labels = []

    all_pr_preds = []
    all_pr_labels = []

    loss_fn = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in dataloader:
            # перенос на device
            batch = {k: v.to(device) for k, v in batch.items()}

            category_labels = batch.pop("category_value")
            priority_labels = batch.pop("priority_value")

            # forward
            cat_logits, pr_logits = model(**batch)

            # loss
            loss_cat = loss_fn(cat_logits, category_labels)
            loss_pr = custom_priority_loss(pr_logits, priority_labels)

            loss = category_coefficient * loss_cat + priority_coefficient * loss_pr

            batch_size = category_labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            # predictions
            cat_preds = torch.argmax(cat_logits, dim=-1)
            pr_preds = torch.argmax(pr_logits, dim=-1)

            # accuracy
            correct_cat += (cat_preds == category_labels).sum().item()
            correct_pr += (pr_preds == priority_labels).sum().item()

            # сохраняем для F1
            all_cat_preds.extend(cat_preds.cpu().tolist())
            all_cat_labels.extend(category_labels.cpu().tolist())

            all_pr_preds.extend(pr_preds.cpu().tolist())
            all_pr_labels.extend(priority_labels.cpu().tolist())

    avg_loss = total_loss / total_samples
    cat_acc = correct_cat / total_samples
    pr_acc = correct_pr / total_samples

    return {
        "loss": avg_loss,
        "category_accuracy": cat_acc,
        "priority_accuracy": pr_acc,
        "cat_preds": all_cat_preds,
        "cat_labels": all_cat_labels,
        "pr_preds": all_pr_preds,
        "pr_labels": all_pr_labels,
    }

In [46]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from statistics import mean

def print_metrics(results):
    print("Loss:", results["loss"])

    print("\n=== Category ===")
    print("Accuracy:", results["category_accuracy"])
    print(classification_report(
        results["cat_labels"],
        results["cat_preds"]
    ))

    print("\n=== Priority ===")
    print("Mean dist:", mean(map(lambda x: abs(x[0] - x[1]), zip(results["pr_labels"], results['pr_preds']))))
    print("Accuracy:", results["priority_accuracy"])
    print(classification_report(
        results["pr_labels"],
        results["pr_preds"]
    ))

    # Confusion Matrix
    cm = confusion_matrix(results["pr_labels"], results["pr_preds"])

    # более читаемый вид
    print("\nConfusion Matrix (formatted):")
    for i, row in enumerate(cm):
        print(f"True {Priority(i).name}\t: {row}")

In [47]:
# 9. Оценка на тестовой выборке
print("\n📊 Оценка на тестовой выборке:")
results = evaluate_model(model, test_loader, device="cuda")
print_metrics(results)


📊 Оценка на тестовой выборке:
Loss: 1.6695349484987225

=== Category ===
Accuracy: 0.9829351535836177
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       114
           1       0.98      1.00      0.99       118
           2       0.97      0.97      0.97       120
           3       0.99      0.99      0.99       109
           4       0.99      0.96      0.98       125

    accuracy                           0.98       586
   macro avg       0.98      0.98      0.98       586
weighted avg       0.98      0.98      0.98       586


=== Priority ===
Mean dist: 0.5546075085324232
Accuracy: 0.5460750853242321
              precision    recall  f1-score   support

           0       0.99      0.47      0.64       157
           1       0.37      0.78      0.50       141
           2       0.52      0.46      0.49       145
           3       0.81      0.48      0.61       143

    accuracy                           0.55       586
   ma

In [319]:
torch.save(model.state_dict(), "model.pt")